# Cotton benchmark — baseline training

Builds a stratified 30k-tile training set from the silver labels, then trains one
model per run. Change `MODEL` at the top and re-run cells 3 onward for each baseline.

**Drive `cotton_bench/` needs:** `cotton_train_pre.zip`, `cotton_train_post.zip`,
`silver_pre/labels.zip`, `silver_post/labels.zip`, `silver_*/raw_preds.json`.

Runtime: L4 or A100 + background execution. ~4–7 h per model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cotton_bench'
!pip install -q ultralytics
import os, json, glob, random, shutil
import numpy as np
for f in ['cotton_train_pre.zip','cotton_train_post.zip',
          'silver_pre/labels.zip','silver_post/labels.zip']:
    print(f, 'OK' if os.path.exists(f'{DRIVE}/{f}') else 'MISSING')

## 1. Build the dataset (run once per session)

In [ ]:
# Unzip tiles and labels
for c in ['pre','post']:
    !unzip -q -o {DRIVE}/cotton_train_{c}.zip -d /content/imgs_{c}
    !unzip -q -o {DRIVE}/silver_{c}/labels.zip -d /content/lbls_{c}
print('pre imgs ', len(glob.glob('/content/imgs_pre/train_tiles/*.jpg')))
print('post imgs', len(glob.glob('/content/imgs_post/train_tiles/*.jpg')))
print('pre lbls ', len(glob.glob('/content/lbls_pre/silver_pre_labels/*.txt')))
print('post lbls', len(glob.glob('/content/lbls_post/silver_post_labels/*.txt')))

In [ ]:
# Stratified 30k subsample: proportional across field x density bin
random.seed(11)
N_TOTAL = 30000

def dbin(n):
    return ('0' if n==0 else '1-20' if n<=20 else '21-60' if n<=60
            else '61-120' if n<=120 else '121-250' if n<=250 else '250+')

pool = {}   # tile -> (field, n_boxes)
for c in ['pre','post']:
    preds = json.load(open(f'{DRIVE}/silver_{c}/raw_preds.json'))
    for t, boxes in preds.items():
        pool[t] = (c, len(boxes))

strata = {}
for t,(c,n) in pool.items():
    strata.setdefault((c, dbin(n)), []).append(t)

sel = []
tot = len(pool)
print(f"{'field':<6}{'bin':<10}{'avail':<9}{'taken'}")
for (c,b), lst in sorted(strata.items()):
    k = max(50, round(N_TOTAL * len(lst)/tot))
    k = min(k, len(lst))
    sel.extend(random.sample(lst, k))
    print(f"{c:<6}{b:<10}{len(lst):<9}{k}")
print('selected tiles:', len(sel))

In [ ]:
# Assemble YOLO folder structure with a val split held out from TRAIN tiles
random.shuffle(sel)
n_val = int(len(sel)*0.05)
splits = {'val': sel[:n_val], 'train': sel[n_val:]}

for s in splits:
    for sub in ['images','labels']:
        os.makedirs(f'/content/ds/{s}/{sub}', exist_ok=True)

missing = 0
for s, tiles in splits.items():
    for t in tiles:
        c = pool[t][0]
        src_i = f'/content/imgs_{c}/train_tiles/{t}'
        src_l = f'/content/lbls_{c}/silver_{c}_labels/{os.path.splitext(t)[0]}.txt'
        if not (os.path.exists(src_i) and os.path.exists(src_l)):
            missing += 1
            continue
        os.symlink(src_i, f'/content/ds/{s}/images/{t}')
        shutil.copy(src_l, f'/content/ds/{s}/labels/')
print('train', len(os.listdir('/content/ds/train/images')),
      'val', len(os.listdir('/content/ds/val/images')), 'missing', missing)

with open('/content/ds/data.yaml','w') as f:
    f.write('path: /content/ds\ntrain: train/images\nval: val/images\n'
            'names:\n  0: cotton_boll\n')

# record exactly which tiles were used (goes in the paper's repro package)
json.dump({s:t for s,t in splits.items()},
          open(f'{DRIVE}/baseline_subset_30k.json','w'))
print('subset list saved to Drive')

## 2. Train — set MODEL, then run

Run once per baseline: `yolov5nu.pt`, `yolo11n.pt`, `yolo11s.pt`, `rtdetr-l.pt`

In [ ]:
MODEL = 'yolov5nu.pt'
RUN   = MODEL.replace('.pt','')
EPOCHS, IMGSZ, BATCH = 100, 640, 16

import os, shutil
from ultralytics import YOLO

!rm -rf /content/runs/{RUN}
m = YOLO(MODEL)
m.train(data='/content/ds/data.yaml', epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        patience=20, project='/content/runs', name=RUN,
        max_det=1000, seed=42, val=True, plots=True,
        save_period=10)

dst = f'{DRIVE}/baselines/{RUN}'
os.makedirs(dst, exist_ok=True)
shutil.copy(f'/content/runs/{RUN}/weights/best.pt', f'{dst}/best.pt')
for f in ['results.csv', 'args.yaml']:
    p = f'/content/runs/{RUN}/{f}'
    if os.path.exists(p): shutil.copy(p, dst)
print('saved to', dst)

In [ ]:
# Save weights + results to Drive
import shutil
dst = f'{DRIVE}/baselines/{RUN}'
os.makedirs(dst, exist_ok=True)
shutil.copy(f'/content/runs/{RUN}/weights/best.pt', f'{dst}/best.pt')
for f in ['results.csv','args.yaml']:
    p = f'/content/runs/{RUN}/{f}'
    if os.path.exists(p): shutil.copy(p, dst)
print('saved to', dst)

import pandas as pd
r = pd.read_csv(f'/content/runs/{RUN}/results.csv')
r.columns = [c.strip() for c in r.columns]
print(r[['epoch','metrics/mAP50(B)','metrics/mAP50-95(B)']].tail(3))

In [ ]:
import os
print(len(os.listdir('/content/ds/train/images')), len(os.listdir('/content/ds/val/images')))
print(os.path.exists('/content/ds/data.yaml'))

In [ ]:
import json, os, shutil

splits = json.load(open(f'{DRIVE}/baseline_subset_30k.json'))
print({k: len(v) for k, v in splits.items()})

!rm -rf /content/ds
for s in splits:
    for sub in ['images', 'labels']:
        os.makedirs(f'/content/ds/{s}/{sub}', exist_ok=True)

missing = 0
for s, tiles in splits.items():
    for t in tiles:
        c = pool[t][0]
        src_i = f'/content/imgs_{c}/train_tiles/{t}'
        src_l = f'/content/lbls_{c}/silver_{c}_labels/{os.path.splitext(t)[0]}.txt'
        if not (os.path.exists(src_i) and os.path.exists(src_l)):
            missing += 1
            continue
        os.symlink(src_i, f'/content/ds/{s}/images/{t}')
        shutil.copy(src_l, f'/content/ds/{s}/labels/')

with open('/content/ds/data.yaml', 'w') as f:
    f.write('path: /content/ds\ntrain: train/images\nval: val/images\n'
            'names:\n  0: cotton_boll\n')

print('train', len(os.listdir('/content/ds/train/images')),
      'val', len(os.listdir('/content/ds/val/images')), 'missing', missing)

In [ ]:
import os
print(os.listdir('/content/runs') if os.path.exists('/content/runs') else 'clean')

In [ ]:
import os
p = '/content/runs/yolov5nu'
print(os.listdir(p))
w = f'{p}/weights'
print(os.listdir(w) if os.path.exists(w) else 'no weights')

In [ ]:
!rm -rf /content/runs/yolov5nu

In [ ]:
import os
print(os.path.exists('/content/ds/data.yaml'),
      len(os.listdir('/content/ds/train/images')),
      len(os.listdir('/content/ds/val/images')))

In [ ]:
!rm -rf /content/runs/rtdetr-l

This is the

In [ ]:
MODEL = 'rtdetr-l.pt'
RUN   = MODEL.replace('.pt','')
EPOCHS, IMGSZ, BATCH = 50, 640, 8

import os, shutil
from ultralytics import YOLO

!rm -rf /content/runs/{RUN}

m = YOLO(MODEL)
m.train(data='/content/ds/data.yaml', epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        patience=20, project='/content/runs', name=RUN,
        max_det=1000, seed=42, val=True, plots=True,
        save_period=5)

dst = f'{DRIVE}/baselines/{RUN}'
os.makedirs(dst, exist_ok=True)
shutil.copy(f'/content/runs/{RUN}/weights/best.pt', f'{dst}/best.pt')
for f in ['results.csv', 'args.yaml']:
    p = f'/content/runs/{RUN}/{f}'
    if os.path.exists(p): shutil.copy(p, dst)
print('saved to', dst)

In [ ]:
!rm -rf /content/runs/rtdetr-l

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/cotton_bench'
for d in sorted(os.listdir(f'{DRIVE}/baselines')):
    print(d, os.listdir(f'{DRIVE}/baselines/{d}'))

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/cotton_bench'
for f in ['gold_v2_verified.zip', 'gold_v2_coco.json', 'gold_v2_tiles.zip',
          'gold_v2_list.txt', 'test_classes.csv']:
    print(f, '✓' if os.path.exists(f'{DRIVE}/{f}') else 'MISSING')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cotton_bench'

import json
for c in ['pre','post']:
    d = json.load(open(f'{DRIVE}/silver_{c}/audit_coco.json'))
    print(c, len(d['images']), 'images |', len(d['annotations']), 'boxes')

In [ ]:
import os, json, glob, random, shutil

CLASS = 'post'
OUT = f'{DRIVE}/silver_{CLASS}'

# tiles must be unzipped; skip if already present
if not os.path.exists('/content/imgs_post'):
    !unzip -q {DRIVE}/cotton_train_post.zip -d /content/imgs_post

preds = json.load(open(f'{OUT}/raw_preds.json'))
print(len(preds), 'post tiles')

random.seed(7)
sample = random.sample(sorted(preds), 150)

# verify these really are post tiles
print('sample check:', sample[0])

!rm -rf /content/audit_post
os.makedirs('/content/audit_post', exist_ok=True)

coco = {'images': [], 'annotations': [],
        'categories': [{'id': 1, 'name': 'cotton_boll'}]}
aid = 1
MAX_SIDE = 120
for iid, t in enumerate(sorted(sample), 1):
    shutil.copy(f'/content/imgs_post/train_tiles/{t}', f'/content/audit_post/{t}')
    coco['images'].append({'id': iid, 'file_name': t, 'width': 640, 'height': 640})
    for x1, y1, x2, y2, c in preds[t]:
        if max(x2-x1, y2-y1) > MAX_SIDE:
            continue
        coco['annotations'].append({'id': aid, 'image_id': iid, 'category_id': 1,
            'bbox': [x1, y1, x2-x1, y2-y1], 'area': (x2-x1)*(y2-y1), 'iscrowd': 0})
        aid += 1

json.dump(coco, open(f'{OUT}/audit_coco_v2.json', 'w'))
shutil.make_archive(f'{OUT}/audit_tiles_v2', 'zip', '/content', 'audit_post')
print(f'{len(sample)} tiles, {aid-1} boxes -> audit_tiles_v2.zip, audit_coco_v2.json')